In [0]:
%sql

SELECT
    COUNT(*) AS linhas_fact_vendas,
    COUNT(DISTINCT id_pedido) AS pedidos_distintos,
    COUNT(DISTINCT id_cliente) AS clientes_distintos,
    COUNT(DISTINCT id_produto) AS produtos_distintos,
    SUM(
        CASE
            WHEN id_pedido IS NULL
              OR id_cliente IS NULL
              OR id_produto IS NULL
            THEN 1 ELSE 0
        END
    ) AS chaves_nulas,
    SUM(
        CASE
            WHEN valor_venda IS NULL
              OR quantidade_vendida IS NULL
            THEN 1 ELSE 0
        END
    ) AS valores_obrigatorios_nulos,
    SUM(
        CASE
            WHEN valor_venda <= 0
              OR quantidade_vendida <= 0
            THEN 1 ELSE 0
        END
    ) AS valores_nao_positivos
FROM first_catalog.default.fact_vendas;

linhas_fact_vendas,pedidos_distintos,clientes_distintos,produtos_distintos,chaves_nulas,valores_obrigatorios_nulos,valores_nao_positivos
51290,25035,1590,10292,0,0,0


In [0]:

%sql
WITH testes AS (

    SELECT
        'Clientes duplicados' AS teste,
        COUNT(*) AS falhas
    FROM (
        SELECT id_cliente
        FROM first_catalog.default.dim_cliente
        GROUP BY id_cliente
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT
        'Produtos duplicados',
        COUNT(*)
    FROM (
        SELECT id_produto
        FROM first_catalog.default.dim_produto
        GROUP BY id_produto
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT
        'Pedidos duplicados',
        COUNT(*)
    FROM (
        SELECT id_pedido
        FROM first_catalog.default.dim_pedido
        GROUP BY id_pedido
        HAVING COUNT(*) > 1
    )

    UNION ALL

    SELECT
        'Vendas sem cliente',
        COUNT(*)
    FROM first_catalog.default.fact_vendas f
    LEFT ANTI JOIN first_catalog.default.dim_cliente c
        ON f.id_cliente = c.id_cliente

    UNION ALL

    SELECT
        'Vendas sem produto',
        COUNT(*)
    FROM first_catalog.default.fact_vendas f
    LEFT ANTI JOIN first_catalog.default.dim_produto p
        ON f.id_produto = p.id_produto

    UNION ALL

    SELECT
        'Vendas sem pedido',
        COUNT(*)
    FROM first_catalog.default.fact_vendas f
    LEFT ANTI JOIN first_catalog.default.dim_pedido p
        ON f.id_pedido = p.id_pedido
)

SELECT teste, falhas
FROM testes
ORDER BY teste;

teste,falhas
Clientes duplicados,0
Pedidos duplicados,0
Produtos duplicados,0
Vendas sem cliente,0
Vendas sem pedido,0
Vendas sem produto,0


In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT id_cliente) AS clientes_distintos
FROM first_catalog.default.dim_cliente;

total_clientes,clientes_distintos
1590,1590


Criar uma tabela de teste

In [0]:
%sql

CREATE OR REPLACE TABLE first_catalog.default.teste_merge_clientes
USING DELTA
AS
SELECT *
FROM first_catalog.default.dim_cliente
WHERE id_cliente IN (
    SELECT id_cliente
    FROM first_catalog.default.dim_cliente
    ORDER BY id_cliente
    LIMIT 3
);

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM first_catalog.default.teste_merge_clientes;

id_cliente,nome_cliente,segmento,cidade,estado,pais,mercado,regiao
AA-10480,Andrew Allen,Consumidor,Bangkok,Bangkok,Thailand,APAC,Southeast Asia
AA-10315,Alex Avila,Consumidor,Round Rock,Texas,United States,US,Central
AA-10375,Allen Armold,Consumidor,Santa Catarina,Nuevo León,Mexico,LATAM,North


Criar uma vista temporária que representa os dados recebidos numa nova carga

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW origem_clientes_incremental AS

SELECT
    id_cliente,
    nome_cliente,
    segmento,
    cidade,
    estado,
    pais,
    mercado,
    regiao
FROM first_catalog.default.teste_merge_clientes

UNION ALL

SELECT
    id_cliente,
    nome_cliente,
    segmento,
    cidade,
    estado,
    pais,
    mercado,
    regiao
FROM first_catalog.default.dim_cliente
WHERE id_cliente = (
    SELECT MAX(id_cliente)
    FROM first_catalog.default.dim_cliente
);

In [0]:

%sql 
MERGE INTO first_catalog.default.teste_merge_clientes AS destino
USING origem_clientes_incremental AS origem
ON destino.id_cliente = origem.id_cliente

WHEN MATCHED THEN UPDATE SET
    destino.nome_cliente = origem.nome_cliente,
    destino.segmento = origem.segmento,
    destino.cidade = origem.cidade,
    destino.estado = origem.estado,
    destino.pais = origem.pais,
    destino.mercado = origem.mercado,
    destino.regiao = origem.regiao

WHEN NOT MATCHED THEN INSERT (
    id_cliente,
    nome_cliente,
    segmento,
    cidade,
    estado,
    pais,
    mercado,
    regiao
)
VALUES (
    origem.id_cliente,
    origem.nome_cliente,
    origem.segmento,
    origem.cidade,
    origem.estado,
    origem.pais,
    origem.mercado,
    origem.regiao
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,3,0,1


Validar o resultado

In [0]:
%sql
SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT id_cliente) AS clientes_distintos
FROM first_catalog.default.teste_merge_clientes;

total_clientes,clientes_distintos
4,4


In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT id_cliente) AS clientes_distintos
FROM first_catalog.default.teste_merge_clientes;

total_clientes,clientes_distintos
4,4


In [0]:

%sql

MERGE INTO first_catalog.default.teste_merge_clientes AS destino
USING origem_clientes_incremental AS origem
ON destino.id_cliente = origem.id_cliente

WHEN MATCHED AND (
    NOT (destino.nome_cliente <=> origem.nome_cliente)
    OR NOT (destino.segmento <=> origem.segmento)
    OR NOT (destino.cidade <=> origem.cidade)
    OR NOT (destino.estado <=> origem.estado)
    OR NOT (destino.pais <=> origem.pais)
    OR NOT (destino.mercado <=> origem.mercado)
    OR NOT (destino.regiao <=> origem.regiao)
)
THEN UPDATE SET
    destino.nome_cliente = origem.nome_cliente,
    destino.segmento = origem.segmento,
    destino.cidade = origem.cidade,
    destino.estado = origem.estado,
    destino.pais = origem.pais,
    destino.mercado = origem.mercado,
    destino.regiao = origem.regiao

WHEN NOT MATCHED THEN INSERT (
    id_cliente,
    nome_cliente,
    segmento,
    cidade,
    estado,
    pais,
    mercado,
    regiao
)
VALUES (
    origem.id_cliente,
    origem.nome_cliente,
    origem.segmento,
    origem.cidade,
    origem.estado,
    origem.pais,
    origem.mercado,
    origem.regiao
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
%sql
SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM first_catalog.default.teste_merge_clientes;

total_clientes,clientes_unicos
4,4


### Criar uma tabela de teste mais realista

In [0]:
%sql

CREATE OR REPLACE TABLE first_catalog.default.teste_merge_dim_cliente
USING DELTA
AS
SELECT *
FROM first_catalog.default.dim_cliente;

num_affected_rows,num_inserted_rows


confirma o número de registos:

In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT id_cliente) AS clientes_unicos
FROM first_catalog.default.teste_merge_dim_cliente;

total_clientes,clientes_unicos
1590,1590


In [0]:
%sql

SELECT
    id_cliente,
    nome_cliente,
    segmento,
    cidade,
    estado,
    pais,
    mercado,
    regiao
FROM first_catalog.default.dim_cliente
ORDER BY id_cliente
LIMIT 5;

id_cliente,nome_cliente,segmento,cidade,estado,pais,mercado,regiao
AA-10315,Alex Avila,Consumidor,Round Rock,Texas,United States,US,Central
AA-10375,Allen Armold,Consumidor,Santa Catarina,Nuevo León,Mexico,LATAM,North
AA-10480,Andrew Allen,Consumidor,Bangkok,Bangkok,Thailand,APAC,Southeast Asia
AA-10645,Anna Andreadi,Consumidor,Jakarta,Jakarta,Indonesia,APAC,Southeast Asia
AA-315,Alex Avila,Consumidor,Hrodna,Hrodna,Belarus,EMEA,EMEA


In [0]:
%sql

CREATE OR REPLACE TEMP VIEW origem_teste_merge AS

-- Simular a alteração de um cliente existente
SELECT
    id_cliente,
    'Alex Avila - TESTE MERGE' AS nome_cliente,
    segmento,
    cidade,
    estado,
    pais,
    mercado,
    regiao
FROM first_catalog.default.teste_merge_dim_cliente
WHERE id_cliente = 'AA-10315'

UNION ALL

-- Simular a chegada de um novo cliente
SELECT
    'ZZ-TEST-MERGE-001' AS id_cliente,
    'Cliente Teste Incremental' AS nome_cliente,
    'Consumidor' AS segmento,
    'Porto' AS cidade,
    'Porto' AS estado,
    'Portugal' AS pais,
    'EU' AS mercado,
    'Teste' AS regiao;

In [0]:
%sql

MERGE INTO first_catalog.default.teste_merge_dim_cliente AS destino
USING origem_teste_merge AS origem
ON destino.id_cliente = origem.id_cliente

WHEN MATCHED AND (
    NOT (destino.nome_cliente <=> origem.nome_cliente)
    OR NOT (destino.segmento <=> origem.segmento)
    OR NOT (destino.cidade <=> origem.cidade)
    OR NOT (destino.estado <=> origem.estado)
    OR NOT (destino.pais <=> origem.pais)
    OR NOT (destino.mercado <=> origem.mercado)
    OR NOT (destino.regiao <=> origem.regiao)
)
THEN UPDATE SET
    destino.nome_cliente = origem.nome_cliente,
    destino.segmento = origem.segmento,
    destino.cidade = origem.cidade,
    destino.estado = origem.estado,
    destino.pais = origem.pais,
    destino.mercado = origem.mercado,
    destino.regiao = origem.regiao

WHEN NOT MATCHED THEN INSERT (
    id_cliente,
    nome_cliente,
    segmento,
    cidade,
    estado,
    pais,
    mercado,
    regiao
)
VALUES (
    origem.id_cliente,
    origem.nome_cliente,
    origem.segmento,
    origem.cidade,
    origem.estado,
    origem.pais,
    origem.mercado,
    origem.regiao
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,1,0,1


Validar o resultado

In [0]:
%sql

SELECT
    COUNT(*) AS total_clientes,
    COUNT(DISTINCT id_cliente) AS clientes_unicos,
    SUM(
        CASE WHEN id_cliente = 'ZZ-TEST-MERGE-001'
             THEN 1 ELSE 0 END
    ) AS cliente_teste_inserido,
    SUM(
        CASE WHEN id_cliente = 'AA-10315'
                  AND nome_cliente = 'Alex Avila - TESTE MERGE'
             THEN 1 ELSE 0 END
    ) AS cliente_teste_atualizado
FROM first_catalog.default.teste_merge_dim_cliente;

total_clientes,clientes_unicos,cliente_teste_inserido,cliente_teste_atualizado
1591,1591,1,1


Limpar os objetos de teste

In [0]:
%sql

DROP VIEW IF EXISTS origem_teste_merge;

DROP TABLE IF EXISTS first_catalog.default.teste_merge_dim_cliente;

DROP TABLE IF EXISTS first_catalog.default.teste_merge_clientes;

DROP VIEW IF EXISTS origem_clientes_incremental;


Validação final das tabelas Gold

In [0]:
%sql

SELECT
    'dim_cliente' AS tabela,
    COUNT(*) AS total_registos,
    COUNT(DISTINCT id_cliente) AS chaves_unicas
FROM first_catalog.default.dim_cliente

UNION ALL

SELECT
    'dim_produto',
    COUNT(*),
    COUNT(DISTINCT id_produto)
FROM first_catalog.default.dim_produto

UNION ALL

SELECT
    'dim_pedido',
    COUNT(*),
    COUNT(DISTINCT id_pedido)
FROM first_catalog.default.dim_pedido

UNION ALL

SELECT
    'fact_vendas',
    COUNT(*),
    COUNT(*)
FROM first_catalog.default.fact_vendas;


tabela,total_registos,chaves_unicas
dim_cliente,1590,1590
dim_produto,10292,10292
dim_pedido,25035,25035
fact_vendas,51290,51290
